In [ ]:
from pyspark.sql import functions as sf
from pyspark.sql import window as sw
from pyspark.sql import types as sdt
from pyspark.sql import SparkSession
from datetime import datetime

LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
STG_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH = "/data/data_files/iceberg/WideWorldImportersDW"

MSSQL_JAR = "C:/data/spark/jars/mssql-jdbc-12.6.5.jre11.jar"

CATALOG_NAME = "local"
STG_CATALOG_NAME = "staging"
WH_CATALOG_NAME = "reporting"

staging_table_name = "staging.Integration.employee_Staging"
wh_table_name = "reporting.dimension.Employees"

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.warehouse", f"file:///{STG_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", f"file:///{RPT_WAREHOUSE_PATH}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()


spark.catalog.setCurrentCatalog(WH_CATALOG_NAME)

spark
spark.sql("SHOW CATALOGS").show(truncate=False)
spark.sql("SHOW NAMESPACES IN reporting").show(truncate=False)
spark.sql("SHOW DATABASES IN reporting").show(truncate=False)
spark.sql("SHOW TABLES IN reporting.dimension").show(truncate=False)
for ns in spark.sql("SHOW NAMESPACES IN reporting").collect():
    namespace = ns["namespace"]
    print(f"\nNamespace: {namespace}")
    spark.sql(f"SHOW TABLES IN reporting.{namespace}").show(truncate=False)


In [8]:
import re

def clean(df):
    new_cols = [re.sub(r'[^0-9A-Za-z]+', '_', c).lower().strip('_') for c in df.columns]
    return df.toDF(*new_cols)

In [9]:
df_sales = spark.table("reporting.Fact.Sale")
df_customer = spark.table("reporting.Dimension.Customer")


df_customer = clean(df_customer)
df_sales = clean(df_sales)

df_customer.printSchema()
df_sales.printSchema()

root
 |-- customer_key: integer (nullable = true)
 |-- wwi_customer_id: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- bill_to_customer: string (nullable = true)
 |-- category: string (nullable = true)
 |-- buying_group: string (nullable = true)
 |-- primary_contact: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- valid_from: timestamp (nullable = true)
 |-- valid_to: timestamp (nullable = true)
 |-- lineage_key: integer (nullable = true)

root
 |-- sale_key: long (nullable = true)
 |-- city_key: integer (nullable = true)
 |-- customer_key: integer (nullable = true)
 |-- bill_to_customer_key: integer (nullable = true)
 |-- stock_item_key: integer (nullable = true)
 |-- invoice_date_key: date (nullable = true)
 |-- delivery_date_key: date (nullable = true)
 |-- salesperson_key: integer (nullable = true)
 |-- wwi_invoice_id: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- package: string (nullable = true)
 |-- qua

In [ ]:
# join facts + dimension
df = df_sales.join(df_customer, "customer_key")

### Create ML features

In [13]:
from pyspark.sql import functions as F

# aggregate to customer level
customer_features = df.groupBy("customer_key").agg(
    F.sum("total_including_tax").alias("total_spent"),
    F.sum("profit").alias("total_profit"),
    F.sum("quantity").alias("total_quantity"),
    F.count("*").alias("num_orders"),
    F.max("invoice_date_key").alias("last_purchase_date")
)

# derive additional features
customer_features = customer_features.withColumn(
    "avg_order_value",
    customer_features.total_spent / customer_features.num_orders
)

customer_features = customer_features.withColumn(
    "recency_days",
    F.datediff(F.current_date(), F.to_date("last_purchase_date"))
)

### Prepare features for ML (VectorAssembler + StandardScaler)

In [14]:
from pyspark.ml.feature import VectorAssembler, StandardScaler


# assemble features into a vector
assembler = VectorAssembler(
    inputCols=[
        "total_spent",
        "total_profit",
        "total_quantity",
        "num_orders",
        "avg_order_value",
        "recency_days"
    ],
    outputCol="features_raw"
)

# transform the data
assembled = assembler.transform(customer_features)

# scale the features
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

# fit and transform the data
scaled = scaler.fit(assembled).transform(assembled)


### KMeans Clustering

In [ ]:
from pyspark.ml.clustering import KMeans


# train KMeans model
kmeans = KMeans(k=4, seed=42, featuresCol="features")

model = kmeans.fit(scaled)

clustered = model.transform(scaled)


### Display cluster results

In [22]:
clustered.select(
    "customer_key",
    "total_spent",
    "num_orders",
    "avg_order_value",
    "recency_days",
    "prediction"
).show(20, truncate=False)


# clustered.printSchema()

+------------+-----------+----------+-----------------+------------+----------+
|customer_key|total_spent|num_orders|avg_order_value  |recency_days|prediction|
+------------+-----------+----------+-----------------+------------+----------+
|148         |1072549.40 |1212      |884.941749174917 |3485        |1         |
|243         |1510113.56 |1448      |1042.896104972376|3476        |1         |
|392         |1360307.16 |1520      |894.938921052632 |3478        |1         |
|31          |1287465.80 |1520      |847.016973684211 |3477        |0         |
|137         |1337421.36 |1364      |980.514193548387 |3480        |1         |
|85          |1210018.48 |1424      |849.732078651685 |3483        |0         |
|251         |935228.96  |1284      |728.371464174455 |3477        |0         |
|65          |1504184.96 |1436      |1047.482562674095|3476        |1         |
|255         |1023499.52 |1348      |759.272640949555 |3483        |0         |
|53          |1240648.80 |1248      |994

### Save model results back to Iceberg

In [18]:
clustered.write.format("iceberg") \
    .mode("overwrite") \
    .saveAsTable("reporting.ml.customer_segments")

UnsupportedOperationException: User-defined types are not supported